# Déjà Code: finding related CodeChef problems

I started with a keyword-based search, then tried sentence embeddings to see whether they find useful matches when the wording is different. After that I compared the rankings, looked at the tags, and explored whether the embeddings form meaningful clusters.

The dataset is the public CodeChef Competitive Programming dataset from [Kaggle](https://www.kaggle.com/datasets/arjoonn/codechef-competitive-programming). I could not verify that it is the exact file intended by the task organizers, so I note that limitation rather than assume it.

In [ ]:
import importlib.util
import subprocess
import sys
import platform

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# Colab already has most of these. Install only if something is missing.
packages = {
    "sentence_transformers": "sentence-transformers",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

missing = [
    package for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", ", ".join(missing))
else:
    print("Required packages are already installed.")

## Load the dataset

The original download is available as a ZIP, but the notebook also accepts the extracted CSV. Upload either `questions.csv` or `questions.csv.zip` when prompted.

In [ ]:
import os
import zipfile
from pathlib import Path
import torch
from google.colab import files

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

uploaded = files.upload()
if not uploaded:
    raise ValueError("Upload questions.csv or questions.csv.zip to continue.")

# Accept the CSV directly or the ZIP downloaded from the dataset page.
upload_dir = Path("/content/deja_code_data")
upload_dir.mkdir(exist_ok=True)

csv_path = None
for filename in uploaded:
    path = Path("/content") / filename
    if path.suffix.lower() == ".csv":
        csv_path = path
        break
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            csv_members = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv") and not name.endswith("/")
            ]
            if not csv_members:
                raise ValueError(f"No CSV file found inside {filename}")
            # The dataset archive is expected to contain questions.csv.
            member = next(
                (name for name in csv_members if Path(name).name.lower() == "questions.csv"),
                csv_members[0]
            )
            extracted = archive.extract(member, upload_dir)
            csv_path = Path(extracted)
            break

if csv_path is None:
    raise ValueError("Please upload a .csv file or a .zip containing a CSV.")

print("Using dataset:", csv_path.name)

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

display(df.head(3))

## A quick look at the data

Before building search, I checked the statement text, tag format, missing fields and duplicates. The raw statements contain some repeated website text and formatting artifacts, so I clean those before creating the search corpus.

In [ ]:
import pandas as pd
import ast
import re
from collections import Counter

# Inspect actual problem statements
for i in range(3):
    row = df.iloc[i]
    print("=" * 80)
    print("QCode:", row["QCode"])
    print("Title:", row["Title"])
    print("Difficulty:", row["level"])
    print("Tags:", row["Tags"])
    print("\nStatement preview:")
    print(str(row["statement"])[:1800])
    print()

# Statement length analysis
df["statement_length"] = df["statement"].astype(str).str.len()

print("\nSTATEMENT LENGTHS")
print(df["statement_length"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
))

print("\nSHORTEST STATEMENTS")
display(
    df.nsmallest(10, "statement_length")[
        ["QCode", "Title", "statement_length", "statement"]
    ]
)

# Duplicate checks
print("\nDUPLICATE CHECKS")
print("Duplicate QCodes:", df["QCode"].duplicated().sum())
print("Duplicate titles:", df["Title"].duplicated().sum())
print("Duplicate statements:", df["statement"].duplicated().sum())

# Inspect tag structure and common tags
def parse_tags(value):
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

tag_lists = df["Tags"].apply(parse_tags)
all_tags = [
    str(tag).strip().lower()
    for tags in tag_lists
    for tag in tags
    if str(tag).strip()
]

print("\nTAG EXAMPLES")
for tags in tag_lists.head(10):
    print(tags)

print("\n30 MOST COMMON TAGS")
print(Counter(all_tags).most_common(30))

In [ ]:
import re
import ast
import html
import pandas as pd

# Keep original data untouched
problems = df.drop(columns=["statement_length"], errors="ignore").copy()

def clean_statement(text):
    text = html.unescape(str(text))

    # Remove repeated CodeChef website boilerplate
    text = re.sub(
        r"All submissions for this problem are available\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Read problems statements in [^\n.]*\.",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove HTML tags while keeping text content
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace without removing meaningful symbols
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)

    return text.strip()

def parse_tags(value):
    try:
        result = ast.literal_eval(str(value))
        if isinstance(result, list):
            return [str(tag).strip().lower() for tag in result]
    except (ValueError, SyntaxError):
        pass
    return []

problems["clean_statement"] = problems["statement"].apply(clean_statement)
problems["parsed_tags"] = problems["Tags"].apply(parse_tags)

# Check duplicates before removal
print("Rows before cleaning:", len(problems))

duplicates = problems.duplicated(
    subset=["clean_statement"], keep="first"
)

print("Duplicate cleaned statements:", duplicates.sum())

# Preserve unique problems
problems = problems.loc[~duplicates].copy()

# Remove empty or near-empty statements
problems = problems[
    problems["clean_statement"].str.len() >= 80
].copy()

problems = problems.reset_index(drop=True)

# Search text: title + problem statement
problems["search_text"] = (
    problems["Title"].fillna("").astype(str)
    + "\n"
    + problems["clean_statement"]
)

print("\nRows after cleaning:", len(problems))
print("Unique QCodes:", problems["QCode"].nunique())
print("Missing search texts:", problems["search_text"].isna().sum())

print("\nExample cleaned statement:")
print(problems.loc[0, "clean_statement"][:1200])

display(
    problems[
        ["QCode", "Title", "level", "parsed_tags", "search_text"]
    ].head()
)

## TF-IDF baseline

TF-IDF is the baseline because it is simple and gives us something to compare against. It tends to work best when the query and a problem statement use similar words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Convert problem statements into TF-IDF vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(problems["search_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))
print("Matrix sparsity:", round(
    100 * (1 - tfidf_matrix.nnz /
          (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])), 2
), "%")


def search_tfidf(query, top_k=5, exclude_qcode=None):
    query_vector = tfidf.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).ravel()

    if exclude_qcode is not None:
        scores[problems["QCode"].eq(exclude_qcode).to_numpy()] = -1

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = problems.iloc[top_indices][
        ["QCode", "Title", "level", "link"]
    ].copy()

    results["similarity"] = scores[top_indices]
    return results.reset_index(drop=True)


# Try a natural-language query
query = """
Given an array of integers, find the maximum sum of a contiguous
subarray. The solution should be efficient for a large input.
"""

display(search_tfidf(query))

## Sentence embeddings

Next I encode each title and cleaned statement with `all-MiniLM-L6-v2`. Cosine similarity between the normalized vectors gives the semantic ranking. The model downloads the first time the notebook runs.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load a pretrained sentence-embedding model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

# Encode the problem statements into dense, normalized vectors
texts = problems["search_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding matrix shape:", embeddings.shape)
print("Embedding data type:", embeddings.dtype)

def search_semantic(query, top_k=5, exclude_qcode=None):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Normalized vectors make dot product equivalent to cosine similarity
    scores = (embeddings @ query_embedding.T).ravel()

    if exclude_qcode is not None:
        scores[problems["QCode"].eq(exclude_qcode).to_numpy()] = -1

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = problems.iloc[top_indices][
        ["QCode", "Title", "level", "link"]
    ].copy()

    results["similarity"] = scores[top_indices]
    return results.reset_index(drop=True)


query = """
Given an array of integers, find the maximum sum of a contiguous
subarray. The solution should be efficient for a large input.
"""

display(search_semantic(query))

## Inspecting the tags

The dataset's tags are not all algorithm concepts: some describe difficulty, contests or other metadata. I use a conservative list of recognizable concepts for the tag-based analysis below. That filter is imperfect, so the tag evaluation is only a proxy for relevance.

In [ ]:
import re
from collections import Counter

# Candidate algorithm/concept tags, based on common patterns
known_non_concept_tags = {
    "easy", "medium", "hard", "beginner",
    "easy-medium", "medium-hard",
    "admin", "admin2", "cakewalk", "challenge",
    "simple", "witua", "div_adm"
}

# Inspect all tags and identify likely concept tags.
# This is a candidate filter, not a perfect classifier.
tag_counts = Counter(
    tag
    for tags in problems["parsed_tags"]
    for tag in tags
)

candidate_tags = {
    tag: count
    for tag, count in tag_counts.items()
    if tag not in known_non_concept_tags
    and not re.fullmatch(r"(cook|april|may|june|july|aug|sep|oct|nov|dec)\d+", tag)
    and not re.fullmatch(r"[a-z0-9_]{5,}", tag)
}

print("Total distinct tags:", len(tag_counts))
print("Candidate concept tags:", len(candidate_tags))

print("\nMost common candidate tags:")
for tag, count in sorted(
    candidate_tags.items(), key=lambda item: item[1], reverse=True
)[:50]:
    print(f"{tag:25} {count}")

print("\nSample problems with candidate tags:")
for _, row in problems.head(15).iterrows():
    concepts = [
        tag for tag in row["parsed_tags"]
        if tag in candidate_tags
    ]
    print(row["QCode"], "|", row["Title"], "|", concepts)

In [ ]:
# A conservative vocabulary of algorithm/data-structure concepts.
# We can expand it after inspecting additional tags.
concept_vocab = {
    "dp", "dynamic-prog", "dp+bitmask",
    "segment-tree", "binary-search", "number-theory",
    "data-structure", "bit", "math", "bfs", "dfs",
    "matrix-expo", "game-theory", "basic-math",
    "fft", "lca", "tree", "heavy-light", "sqrt-decomp",
    "suffix-array", "gcd", "bitwise-operatn",
    "gauss-elim", "shortest-path", "union-find",
    "disjoint-set", "inclusn-exclusn", "xor",
    "divide-and-conq", "trie", "kmp", "convex-hull",
    "lcm", "line-sweep", "lcp", "prefix-sum",
    "min-cost-flow", "two-pointers", "sprague-grundy",
    "mst", "sorting", "greedy", "graph", "geometry",
    "combinatorics", "probability", "permutation",
    "string", "strings", "modular-arithmetic",
    "topological-sort", "flow", "matching",
    "hashing", "rolling-hash", "recursion",
    "backtracking", "implementation", "simulation",
    "stack", "queue", "heap", "priority-queue",
    "linked-list", "binary-tree", "fenwick-tree",
    "binary-indexed-tree", "minimum-spanning-tree",
    "dijkstra", "floyd-warshall", "bellman-ford",
    "strongly-connected-components", "scc",
    "articulation-points", "bridges", "diameter",
    "centroid-decomposition", "persistent-segment-tree",
    "lazy-propagation", "range-query", "range-minimum-query",
    "manacher", "z-algorithm", "suffix-array",
    "suffix-automaton", "aho-corasick", "palindrome",
    "knapsack", "longest-common-subsequence",
    "longest-increasing-subsequence", "maximum-flow",
    "minimax", "nim", "grundy", "mex",
    "matrix", "linear-algebra", "expected-value",
    "inclusion-exclusion", "modular", "exponentiation",
    "fast-exponentiation", "prime", "sieve",
    "factorization", "euclid", "extended-euclid",
    "coordinate-compression", "sweep-line",
    "meet-in-the-middle", "bitmask", "bitmask-dp",
    "two-pointer", "sliding-window", "prefix-sums",
    "difference-array", "intervals", "interval-scheduling"
}

def verified_concept_tags(tags):
    return sorted(set(tags) & concept_vocab)

problems["concept_tags"] = problems["parsed_tags"].apply(
    verified_concept_tags
)

tagged = problems[problems["concept_tags"].map(len) > 0]

print("Problems with at least one allowlisted concept tag:",
      len(tagged), "/", len(problems))

print("\nMost common allowlisted concepts:")
from collections import Counter

concept_counts = Counter(
    tag for tags in problems["concept_tags"] for tag in tags
)

for tag, count in concept_counts.most_common(30):
    print(f"{tag:30} {count}")

print("\nSample tagged problems:")
display(
    tagged[["QCode", "Title", "concept_tags"]].head(25)
)

In [ ]:
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Map known aliases to canonical concept labels
tag_aliases = {
    "dynamic-prog": "dp",
    "dp+bitmask": "bitmask-dp",
    "disjoint-set": "union-find",
    "binary-indexed-tree": "fenwick-tree",
    "minimum-spanning-tree": "mst",
    "two-pointer": "two-pointers",
    "prefix-sums": "prefix-sum",
    "sweep-line": "line-sweep",
    "modular-arithmetic": "number-theory",
    "basic-math": "math",
    "strings": "string",
}

def normalize_concepts(tags):
    normalized = {
        tag_aliases.get(tag, tag)
        for tag in tags
    }
    return sorted(normalized)

problems["normalized_tags"] = problems["concept_tags"].apply(
    normalize_concepts
)

# Build an inverted index: concept -> problem indices
concept_to_indices = defaultdict(set)

for idx, tags in enumerate(problems["normalized_tags"]):
    for tag in tags:
        concept_to_indices[tag].add(idx)

# Keep only concepts with enough examples to support evaluation
usable_concepts = {
    tag: indices
    for tag, indices in concept_to_indices.items()
    if len(indices) >= 5
}

print("Usable concepts (at least 5 problems):", len(usable_concepts))
print("\nMost frequent normalized concepts:")

for tag, indices in sorted(
    usable_concepts.items(),
    key=lambda item: len(item[1]),
    reverse=True
)[:25]:
    print(f"{tag:25} {len(indices)} problems")

# Build candidate evaluation queries with at least one relevant alternative
eligible_queries = []

for idx, tags in enumerate(problems["normalized_tags"]):
    relevant = set()

    for tag in tags:
        relevant.update(concept_to_indices.get(tag, set()))

    relevant.discard(idx)

    if relevant:
        eligible_queries.append({
            "query_idx": idx,
            "relevant_indices": relevant
        })

print("\nEligible query problems:", len(eligible_queries))
print("Example evaluation case:")

if eligible_queries:
    example = eligible_queries[0]
    query_idx = example["query_idx"]

    print("Query:", problems.iloc[query_idx]["QCode"],
          "-", problems.iloc[query_idx]["Title"])
    print("Tags:", problems.iloc[query_idx]["normalized_tags"])
    print("Potential relevant examples:")

    relevant_indices = list(example["relevant_indices"])[:5]
    display(
        problems.iloc[relevant_indices][
            ["QCode", "Title", "normalized_tags"]
        ]
    )

## A tag-overlap evaluation

For this experiment, another problem is considered potentially relevant when it shares at least one normalized concept tag with the query problem. I sample up to 200 eligible problems and compare TF-IDF with the embedding ranking.

This is not a manually verified ground-truth benchmark. Shared tags can be too broad, and relevant problems can have missing or different tags, so read these metrics as a rough diagnostic rather than a final measure of search quality.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Use the same query set and relevance candidates for both systems.
# Only queries with at least one relevant alternative are included.
rng = np.random.default_rng(42)
selected = rng.choice(
    len(eligible_queries),
    size=min(200, len(eligible_queries)),
    replace=False
)

eval_cases = [eligible_queries[i] for i in selected]

def evaluate_retriever(method, k=5):
    precisions = []
    recalls = []
    reciprocal_ranks = []
    rows = []

    for case in eval_cases:
        query_idx = case["query_idx"]
        relevant = case["relevant_indices"]

        if method == "tfidf":
            scores = (
                tfidf_matrix[query_idx] @ tfidf_matrix.T
            ).toarray().ravel()
        else:
            scores = embeddings @ embeddings[query_idx]

        # Exclude the query problem itself
        scores[query_idx] = -np.inf

        top_indices = np.argsort(scores)[::-1][:k]
        hits = [
            idx for idx in top_indices
            if idx in relevant
        ]

        precision = len(hits) / k
        recall = len(hits) / len(relevant)

        ranks = [
            rank for rank, idx in enumerate(top_indices, start=1)
            if idx in relevant
        ]
        reciprocal_rank = 1 / min(ranks) if ranks else 0.0

        precisions.append(precision)
        recalls.append(recall)
        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "QCode": problems.iloc[query_idx]["QCode"],
            "Title": problems.iloc[query_idx]["Title"],
            "Relevant_count": len(relevant),
            "Hits@5": len(hits),
            "Precision@5": precision,
            "Recall@5": recall,
            "Reciprocal_Rank": reciprocal_rank
        })

    summary = {
        "Model": method,
        "Queries": len(eval_cases),
        "Precision@5": np.mean(precisions),
        "Recall@5": np.mean(recalls),
        "MRR": np.mean(reciprocal_ranks)
    }

    return summary, pd.DataFrame(rows)

tfidf_summary, tfidf_details = evaluate_retriever("tfidf")
semantic_summary, semantic_details = evaluate_retriever("semantic")

comparison = pd.DataFrame([tfidf_summary, semantic_summary])
display(comparison.round(4))

print("Example evaluation cases:")
display(semantic_details.head(10))

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Prefer problems with at least one concept tag
tagged_indices = [
    i for i, tags in enumerate(problems["normalized_tags"])
    if tags
]

audit_indices = rng.choice(
    tagged_indices,
    size=min(10, len(tagged_indices)),
    replace=False
)

audit_rows = []

for query_idx in audit_indices:
    query_row = problems.iloc[query_idx]

    # TF-IDF results
    tfidf_scores = (
        tfidf_matrix[query_idx] @ tfidf_matrix.T
    ).toarray().ravel()
    tfidf_scores[query_idx] = -np.inf
    tfidf_top = np.argsort(tfidf_scores)[::-1][:3]

    # Semantic results
    semantic_scores = embeddings @ embeddings[query_idx]
    semantic_scores[query_idx] = -np.inf
    semantic_top = np.argsort(semantic_scores)[::-1][:3]

    for method, indices, scores in [
        ("TF-IDF", tfidf_top, tfidf_scores),
        ("Semantic", semantic_top, semantic_scores)
    ]:
        for rank, result_idx in enumerate(indices, start=1):
            result = problems.iloc[result_idx]

            audit_rows.append({
                "Query": query_row["QCode"],
                "Query title": query_row["Title"],
                "Query tags": ", ".join(query_row["normalized_tags"]),
                "Model": method,
                "Rank": rank,
                "Result QCode": result["QCode"],
                "Result title": result["Title"],
                "Result tags": ", ".join(result["normalized_tags"]),
                "Similarity": round(float(scores[result_idx]), 4)
            })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

## Hybrid search

The hybrid score combines semantic similarity with a normalized TF-IDF score. I use a semantic weight of 0.65 and lexical weight of 0.35 for this experiment. It is a simple baseline combination, not a learned ranker, so it may not improve every query.

In [ ]:

import numpy as np
import pandas as pd

def search_hybrid(query, top_k=5, semantic_weight=0.65,
                  lexical_weight=0.35):
    semantic_scores = (
        model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0] @ embeddings.T
    )

    lexical_scores = (
        tfidf.transform([query]) @ tfidf_matrix.T
    ).toarray().ravel()

    # Normalize lexical scores to a comparable 0–1 range.
    max_lexical = lexical_scores.max()
    if max_lexical > 0:
        lexical_scores = lexical_scores / max_lexical

    combined_scores = (
        semantic_weight * semantic_scores
        + lexical_weight * lexical_scores
    )

    ranked = np.argsort(combined_scores)[::-1]

    results = []
    for idx in ranked:
        results.append({
            "QCode": problems.iloc[idx]["QCode"],
            "Title": problems.iloc[idx]["Title"],
            "Tags": ", ".join(problems.iloc[idx]["normalized_tags"]),
            "Semantic score": round(float(semantic_scores[idx]), 4),
            "Lexical score": round(float(lexical_scores[idx]), 4),
            "Hybrid score": round(float(combined_scores[idx]), 4)
        })
        if len(results) == top_k:
            break

    return pd.DataFrame(results)

test_query = (
    "Given an array of integers, find the maximum sum "
    "of a contiguous subarray efficiently."
)

display(search_hybrid(test_query))


## Try some natural-language queries

These examples cover arrays, graph paths, range queries, strings and counting ways. Comparing all three methods makes it easier to spot cases where word overlap helps, or where semantic matching finds a better candidate.

In [ ]:

manual_queries = [
    {
        "query": "Find the maximum sum of a contiguous subarray.",
        "concept": "Maximum subarray / prefix sums / dynamic programming"
    },
    {
        "query": "Find the shortest path between vertices in a weighted graph.",
        "concept": "Graph shortest paths"
    },
    {
        "query": "Answer range sum queries efficiently on an array.",
        "concept": "Prefix sums / Fenwick tree / segment tree"
    },
    {
        "query": "Determine whether a string is a palindrome.",
        "concept": "String algorithms / palindrome"
    },
    {
        "query": "Find the number of ways to reach a target using smaller choices.",
        "concept": "Dynamic programming / combinatorics"
    }
]

for i, item in enumerate(manual_queries, start=1):
    print(f"\nQUERY {i}: {item['query']}")
    print("Expected concept:", item["concept"])

    print("\nTF-IDF:")
    display(search_tfidf(item["query"], top_k=5))

    print("\nSemantic:")
    display(search_semantic(item["query"], top_k=5))

    print("\nHybrid:")
    display(search_hybrid(item["query"], top_k=5))


## Clustering the embeddings

I first compared a few values of *k* rather than assuming the right number of clusters. The silhouette score is low in these experiments, so the clusters should be treated as exploratory groupings, not reliable algorithm labels.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 1. Cluster the sentence embeddings.
n_clusters = 8

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(embeddings)
problems["cluster"] = cluster_labels

# 2. Evaluate the clustering.
silhouette = silhouette_score(
    embeddings,
    cluster_labels,
    sample_size=min(1000, len(embeddings)),
    random_state=42
)

print("Number of problems:", len(problems))
print("Number of clusters:", n_clusters)
print("Silhouette score:", round(silhouette, 4))

# 3. Reduce embeddings to 2D for visualization.
pca = PCA(n_components=2, random_state=42)
coordinates = pca.fit_transform(embeddings)

print(
    "Variance explained by first two PCA components:",
    round(pca.explained_variance_ratio_.sum(), 4)
)

# 4. Plot the clusters.
plt.figure(figsize=(11, 7))

scatter = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=cluster_labels,
    cmap="tab10",
    s=18,
    alpha=0.7
)

plt.title("CodeChef Problem Clusters Using Sentence Embeddings")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

# 5. Inspect representative problems from each cluster.
for cluster_id in range(n_clusters):
    members = problems[problems["cluster"] == cluster_id]

    print(f"\nCluster {cluster_id} — {len(members)} problems")
    display(
        members[
            ["QCode", "Title", "normalized_tags"]
        ].head(10)
    )


In [ ]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

cluster_results = []

for k in [4, 6, 8, 10, 12, 15]:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = km.fit_predict(embeddings)

    score = silhouette_score(
        embeddings,
        labels,
        sample_size=min(1000, len(embeddings)),
        random_state=42
    )

    sizes = pd.Series(labels).value_counts()

    cluster_results.append({
        "Clusters": k,
        "Silhouette score": round(score, 4),
        "Smallest cluster": int(sizes.min()),
        "Largest cluster": int(sizes.max())
    })

cluster_comparison = pd.DataFrame(cluster_results)
display(
    cluster_comparison.sort_values(
        "Silhouette score",
        ascending=False
    )
)


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Final exploratory clustering configuration
final_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

problems["cluster"] = final_kmeans.fit_predict(embeddings)

final_silhouette = silhouette_score(
    embeddings,
    problems["cluster"],
    sample_size=min(1000, len(embeddings)),
    random_state=42
)

print("Final number of clusters: 4")
print("Silhouette score:", round(final_silhouette, 4))
print("\nCluster sizes:")
print(problems["cluster"].value_counts().sort_index())

# PCA visualization
pca = PCA(n_components=2, random_state=42)
coordinates = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=problems["cluster"],
    cmap="tab10",
    s=20,
    alpha=0.7
)

plt.title("Exploratory Clustering of CodeChef Problems")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.show()

# Inspect 12 sample problems per cluster
for cluster_id in range(4):
    members = problems[problems["cluster"] == cluster_id]

    print(f"\nCluster {cluster_id}: {len(members)} problems")
    display(
        members[
            ["QCode", "Title", "normalized_tags"]
        ].sample(
            n=min(12, len(members)),
            random_state=42
        ).sort_values("Title")
    )


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

# Find terms that are especially prominent within each cluster.
cluster_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3
)

cluster_text_matrix = cluster_vectorizer.fit_transform(
    problems["search_text"]
)
feature_names = np.array(cluster_vectorizer.get_feature_names_out())

for cluster_id in sorted(problems["cluster"].unique()):
    indices = np.flatnonzero(
        problems["cluster"].to_numpy() == cluster_id
    )

    mean_scores = np.asarray(
        cluster_text_matrix[indices].mean(axis=0)
    ).ravel()

    top_indices = mean_scores.argsort()[-12:][::-1]

    print(f"\nCluster {cluster_id}")
    print("Distinctive terms:", ", ".join(feature_names[top_indices]))

    display(
        problems.loc[indices, ["QCode", "Title", "normalized_tags"]]
        .head(8)
    )


I also tested a more aggressive text-cleaning approach for clustering. This is a small experiment to check whether removing common statement words helps; the score needs to be compared with the earlier result before deciding whether to keep it.

In [ ]:

import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Remove common competitive-programming boilerplate.
extra_stopwords = {
    "chef", "input", "output", "test", "tests", "case",
    "cases", "line", "lines", "number", "numbers",
    "integer", "integers", "contains", "given", "print",
    "find", "solution", "problem", "statement"
}

def clean_for_clustering(text):
    text = str(text).lower()
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = [
        word for word in text.split()
        if word not in ENGLISH_STOP_WORDS
        and word not in extra_stopwords
        and len(word) > 2
    ]
    return " ".join(words)

cluster_text = (
    problems["Title"].fillna("") + " " +
    problems["clean_statement"].fillna("")
).apply(clean_for_clustering).tolist()

cluster_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

clean_embeddings = cluster_model.encode(
    cluster_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

clean_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)
clean_labels = clean_kmeans.fit_predict(clean_embeddings)

clean_silhouette = silhouette_score(
    clean_embeddings,
    clean_labels,
    sample_size=min(1000, len(clean_embeddings)),
    random_state=42
)

print("Original silhouette:", round(0.0404, 4))
print("Cleaned-text silhouette:", round(clean_silhouette, 4))


## Inspecting the clusters

The next cells show sample problems and common tags from each cluster. A cluster may contain several topics, so I avoid assigning a precise topic label unless the examples support it.

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Cluster the problem statement embeddings
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

problems["cluster"] = cluster_labels

# Reduce the 384-dimensional embeddings to 2 dimensions for visualization
pca = PCA(n_components=2, random_state=42)
points_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    points_2d[:, 0],
    points_2d[:, 1],
    c=cluster_labels,
    cmap="viridis",
    s=18,
    alpha=0.7
)

plt.title("Exploratory Clustering of CodeChef Problems")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.show()

print("Silhouette score:", round(
    silhouette_score(embeddings, cluster_labels), 4
))
print(
    "Variance explained by first two PCA components:",
    round(pca.explained_variance_ratio_.sum() * 100, 2), "%"
)
print("\nCluster sizes:")
print(problems["cluster"].value_counts().sort_index())

from collections import Counter

for cluster_id in sorted(problems["cluster"].unique()):
    subset = problems[problems["cluster"] == cluster_id]

    tag_counts = Counter(
        tag
        for tags in subset["normalized_tags"]
        if isinstance(tags, list)
        for tag in tags
    )

    print(f"\n{'=' * 55}")
    print(f"CLUSTER {cluster_id} — {len(subset)} problems")
    print("=" * 55)
    print("Most common concept tags:")
    print(tag_counts.most_common(12))

    print("\nSample problems:")
    for _, row in subset.sample(
        min(12, len(subset)), random_state=42
    ).iterrows():
        print(f"- {row['QCode']}: {row['Title']}")


In [ ]:

for cluster_id in sorted(problems["cluster"].unique()):
    subset = problems[problems["cluster"] == cluster_id]

    print(f"\n{'=' * 65}")
    print(f"CLUSTER {cluster_id} — {len(subset)} problems")
    print("=" * 65)

    for _, row in subset.head(8).iterrows():
        print(f"{row['QCode']} | {row['Title']}")
        tags = row.get("normalized_tags", [])
        print("Tags:", ", ".join(tags[:6]) if isinstance(tags, list) else tags)


In [ ]:

from collections import Counter

for cluster_id in sorted(problems["cluster"].unique()):
    subset = problems[problems["cluster"] == cluster_id]

    tag_counts = Counter(
        tag
        for tags in subset["normalized_tags"]
        if isinstance(tags, list)
        for tag in tags
    )

    print(f"\n{'=' * 55}")
    print(f"CLUSTER {cluster_id} — {len(subset)} problems")
    print("=" * 55)
    print("Most common concept tags:")
    print(tag_counts.most_common(12))

    print("\nSample problems:")
    for _, row in subset.sample(
        min(12, len(subset)), random_state=42
    ).iterrows():
        print(f"- {row['QCode']}: {row['Title']}")


## False similarities

A high score does not guarantee that two problems ask for the same thing. These examples are useful because the wording overlaps, but the objective differs. I checked the statements rather than judging only by the titles.

In [ ]:

false_similarity_cases = [
    {
        "query": "Find the maximum sum of a contiguous subarray.",
        "qcode": "MXSUBARR",
        "expected": "Maximum-sum contiguous subarray",
        "issue": "Disjoint subarrays are not necessarily the same objective."
    },
    {
        "query": "Find the shortest path between vertices in a weighted graph.",
        "qcode": "RG_01",
        "expected": "Minimum total path distance",
        "issue": "Maximizing the minimum edge weight is a different path objective."
    },
    {
        "query": "Answer range sum queries efficiently on an array.",
        "qcode": "RRSUM",
        "expected": "Sum of values over array ranges",
        "issue": "Counting pairs that sum to a target is a different task."
    }
]

for case in false_similarity_cases:
    match = problems[problems["QCode"] == case["qcode"]]

    print("\n" + "=" * 70)
    print("Query:", case["query"])
    print("Problem:", case["qcode"])
    print("Expected intent:", case["expected"])
    print("Potential false similarity:", case["issue"])

    if not match.empty:
        row = match.iloc[0]
        print("Title:", row["Title"])
        print("Statement:", str(row["clean_statement"])[:900])
    else:
        print("Problem code not found in cleaned dataset.")


## Final check: compare the top results

This last cell prints the top five results from each method for the same set of queries. The results can change slightly with library/model versions, and similarity scores should be used to rank candidates—not interpreted as probabilities of correctness.

In [ ]:

test_queries = [
    "Find the maximum sum of a contiguous subarray.",
    "Find the shortest path between vertices in a weighted graph.",
    "Answer range sum queries efficiently on an array.",
    "Determine whether a string is a palindrome.",
    "Find the number of ways to reach a target using smaller choices."
]

for query in test_queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)

    for method_name, search_fn in [
        ("TF-IDF", search_tfidf),
        ("Semantic", search_semantic),
        ("Hybrid", search_hybrid)
    ]:
        print(f"\n{method_name} — Top 5")
        results = search_fn(query, top_k=5)

        # Supports either DataFrame or list-style search results
        if isinstance(results, pd.DataFrame):
            display_cols = [
                c for c in ["QCode", "Title", "score", "similarity"]
                if c in results.columns
            ]
            display(results[display_cols])
        else:
            print(results)
